# RF Testing

Checking path so that I know which qickdawg it's using. It should be the one in the local folder and not the github 

In [26]:
import sys
print(sys.path)

['/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/home/artiq-development/artiq-master/repository/connecting_to_rfsoc/.venv/lib/python3.12/site-packages', '/home/artiq-development/artiq-master/repository/connecting_to_rfsoc/qick-dawg/src']


In [27]:
!pip show qickdawg

Name: qickdawg
Version: 1.2.1
Summary: Software for full quantum control of nitrogen-vacancy defects and other quantum defects in diamond
Home-page: 
Author: 
Author-email: Andy Mounce <amounce@sandia.gov>, Emmeline Riendeau <eriendeau@uchicago.edu>
License: MIT License 

Copyright 2023 National Technology & Engineering Solutions of Sandia, LLC (NTESS). Under the terms of Contract DE-NA0003525 with NTESS, the U.S. Government retains certain rights in this software.

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substa

In [28]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from copy import copy
import qickdawg as qd

from scipy.optimize import curve_fit
from scipy.signal import find_peaks

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
qd.start_client('128.95.31.218')

QICK library version mismatch: 0.2.324 remote (the board), 0.2.302 local (the PC)
                        This may cause errors, usually KeyError in QickConfig initialization.
                        If this happens, you must bring your versions in sync.


## Load Default Configuration

In [30]:
default_config = qd.NVConfiguration()

default_config.adc_channel = 0
default_config.edge_counting = True
default_config.high_threshold = 2000
default_config.low_threshold = 500


default_config.mw_channel = 0
default_config.mw_nqz = 1
default_config.mw_gain = 5000

default_config.laser_gate_pmod = 0

default_config.relax_delay_tns = 50 # between each rep, wait for everything to catch up, mostly aom


# Simple RF Pulse

In [59]:
from qickdawg.nvpulsing.rfpulse import RFPulse

soc = qd.soc
config = copy(default_config)

config.readout_integration_tus = qd.max_int_time_tus # necessary but not used
config.mw_fMHz = 500# frequency; fMHz, freg, fGHz
config.pulse_len_tns = 200 # pulse duration; tus, tns, treg
config.relax_delay_tns = 20 # 1.62 us
config.gain = 30000 # up to 32767 
config.reps = 1
config.num_pulses = 1

config.init_delay_tns = 0# wait before starting the sequence
prog = RFPulse(config)
prog.run_rounds(soc,rounds = 1, start_src="external")

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:03<00:00,  3.05s/it]


# CPMGXY8 
Simple CPMGXY for investigating looping in body and init

In [ ]:
from qickdawg.nvpulsing.cpmgxy_test import CPMGXY8nDelaySweepInBody

soc = qd.soc
config = copy(default_config)

config.mw_gain = 30000
config.mw_fMHz = 500
config.mw_pi2_tns = 100

config.relax_delay_tus = 0.05

config.scaling_mode = 'linear'
config.delay_start_tns = 50
config.delay_end_tns = 100
config.nsweep_points = 1 # >0 or division by 0 error 

config.reps=1
config.n_cpmg = 1
prog = CPMGXY8nDelaySweepInBody(config)
prog.run_rounds(soc, rounds=0, start_src="external")

In [72]:
delays = np.linspace(50, 100, 2)
print(delays.size)

2


# CPMGXY8 Sweep Param outside

In [ ]:
from qickdawg.nvpulsing.cpmgxy_sweep_outside import CPMGXY8nDelaySweepOutside

delays = np.linspace(50, 100, 2)
for i in range(delays.size):
    soc = qd.soc
    config = copy(default_config)

    config.mw_gain = 30000
    config.mw_fMHz = 500
    config.mw_pi2_tns = 100

    config.relax_delay_tus = 0.5
    config.delay_tns = delays[i]

    config.reps=1
    config.n_cpmg = 1
    prog = CPMGXY8nDelaySweepOutside(config)
    prog.run_rounds(soc, rounds=1, start_src="external")

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:01<00:00,  1.44it/s]


# Gain Sweep (viewable on oscilliscope)

pmod channel width not tunable when time is very short it seems.

In [17]:
from qickdawg.nvpulsing.rftest import RFTest 

config = copy(default_config)

config.readout_integration_tus = qd.max_int_time_tus # necessary but not used
config.mw_fMHz = 200 # frequency; fMHz, freg, fGHz
config.pulse_len_tns = 100 # pulse duration; tus, tns, treg
config.relax_delay_tns = 20 # 1.62 us
config.trigger_width_tns = 100

config.add_unitless_linear_sweep("gain", 0, 30000, delta=10)
config.pre_init = False # not sure if necessary
config.reps = 1

config.repitition = 1

prog = RFTest(config)
_ = prog.acquire()


Requested 0 to 30000 by 10
Instead using 0 to 30010 by 10 in 3001


## Envelopes!

In [12]:
# Get the maxv for envelopes on the mw_channel
default_config.soccfg.get_maxv(default_config.mw_channel)
# maxv is tied to gain units
# if maxv isn't declared then the envelope will use the maximum as default

32766

In [10]:
from qickdawg.testfunctions.rftest_envelope import RFTest_Envelope

config = copy(default_config)

config.readout_integration_tus = qd.max_int_time_tus # necessary but not used
config.mw_fMHz = 2700 # frequency; fMHz, freg, fGHz
config.pulse_len_tns = 300 # pulse duration; tus, tns, treg
config.pulse_sigma_tns = 100 # sigma for gaussian pulse
config.relax_delay_tns = 500 # 1.62 us
config.trigger_width_tns = 500

config.add_unitless_linear_sweep("gain", 32000, 31900, delta=-1)
config.pre_init = False # not sure if necessary
config.reps = 1
config.repitition = 1

prog = RFTest_Envelope(config)
_ = prog.acquire()


Requested 32000 to 31900 by -1
Instead using 32000 to 31899 by -1 in 101
